<a href="https://colab.research.google.com/github/KjelleJ/enkla-ai-experiment/blob/main/AIX_2A_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
#Modell: intfloat/multilingual-e5-small för inbäddning av en mening

### 100 miljoner parametrar i 12 lager. Modellen laddas från Hugging Face.
---

In [ ]:
# Vi kör utan nyckel. Du får ett meddelande med en varning. Välj Cancel...
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

In [ ]:
# här kan du ändra eller lägga till text
texter = [
    "AI kan användas för att analysera stora mängder text.",
    "Artificiell intelligens kan hjälpa oss att bearbeta mycket text.",
    "The cat is sitting on the carpet.",
    "Katten sitter på mattan."
]


In [ ]:
# beräkna inbäddningar
emb = model.encode(texter,normalize_embeddings=True)

print(emb.shape)


#Visualisering
###Våra vektorer är 384d. För att kunna plotta i 2d (planet) omvandlar vi vektorerna från 384d till 2d med hjälp av PCA. Då kan vi se att "liknande" meningar samlas i kluster.

In [ ]:
from sklearn.decomposition import PCA
from matplotlib import pyplot

pca = PCA(n_components=2)
result = pca.fit_transform(emb)
pyplot.scatter(result[:, 0], result[:, 1])
for i in range(len(emb)):
    pyplot.annotate("  " + str(i), xy=(result[i, 0], result[i, 1]))
pyplot.show()
for i in range(len(emb)):
    print(f"{i}: {texter[i]}")

---
#Vektor-databas - överkurs
###Vår modell är tränad för att söka i vektordatabaser. Ange "passage:" för en mening vars inbäddning finns i databasen. "query:" för en mening vars inbäddning söks i databasen.

In [ ]:
# query-texter: inbäddning används för fråga mot databasen
q_texter = ["query: " + t for t in texter]
q_texter

In [ ]:
# passage-texter: inbäddning finns i databasen
p_texter = [t.replace("query:", "passage:") for t in q_texter]
p_texter

In [ ]:
# beräkna inbäddningar
q_emb = model.encode(q_texter, normalize_embeddings=True)
p_emb = model.encode(p_texter, normalize_embeddings=True)

In [ ]:
# funktion för att beräkna cosine similarity

import numpy as np

def cossim(v1, v2):
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    # Undvik division med noll om någon av vektorerna är en nollvektor
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0

    return dot_product / (norm_v1 * norm_v2)

In [ ]:
# Testa funktionen med två exempelvektorer
a = np.array([1, 2, 3])
b = np.array([2, 4, 6]) # Helt parallell vektor (bör ge 1.0)
c = np.array([-1, -2, -3]) # Helt motriktad vektor (bör ge -1.0)

print(f"Likhet (a, b): {cossim(a, b):.4f}")
print(f"Likhet (a, c): {cossim(a, c):.4f}")

In [ ]:
# beräkna likhet mellan q- och p-vektorer
sim = []
for i in range(len(q_emb)):
    for j in range(len(p_emb)):
        dot_product = cossim(q_emb[i], p_emb[j])
        txt = f"q{i} p{j} {dot_product:.4f}"
        sim.append(txt)
        print(txt)

In [ ]:
# Sortera sim-listan efter det tredje fältet i fallande ordning
sim_sorted = sorted(sim, key=lambda x: float(x.split()[2]), reverse=True)
sim_sorted

In [ ]:
# Visualisera
from sklearn.decomposition import PCA
from matplotlib import pyplot

pca = PCA(n_components=2)
result = pca.fit_transform(q_emb)
pyplot.scatter(result[:, 0], result[:, 1], c='green')
for i in range(len(q_emb)):
    pyplot.annotate("  " + str(i), xy=(result[i, 0], result[i, 1]))

pca = PCA(n_components=2)
result = pca.fit_transform(p_emb)
pyplot.scatter(result[:, 0], result[:, 1], c='red')
for i in range(len(p_emb)):
    pyplot.annotate("  " + str(i), xy=(result[i, 0], result[i, 1]))

pyplot.legend(['query', 'passage'])

pyplot.show()
for i in range(len(emb)):
    print(f"{i}: {texter[i]}")